In [21]:
import tensorflow as tf
from datasets import load_dataset
import numpy as np
from tqdm import tqdm

data_dir = '/n/holylabs/LABS/albergo_lab/Everyone/nmboffi/datasets'

# Download from HuggingFace
print("Downloading AFHQ from HuggingFace...")
dataset = load_dataset("huggan/afhqv2", cache_dir=data_dir)

def to_tf_dataset(hf_dataset, split):
    """Convert HuggingFace dataset to TF dataset."""
    def generator():
        for example in hf_dataset[split]:
            image = np.array(example['image'])
            label = example['label']
            yield {'image': image, 'label': label}  # Changed to dict
    
    return tf.data.Dataset.from_generator(
        generator,
        output_signature={
            'image': tf.TensorSpec(shape=(512, 512, 3), dtype=tf.uint8),
            'label': tf.TensorSpec(shape=(), dtype=tf.int32)
        }
    )

# Convert to TF datasets
ds_train = to_tf_dataset(dataset, 'train')

dataset_infos.json:   0%|          | 0.00/827 [00:00<?, ?B/s]

train-00000-of-00013.parquet:   0%|          | 0.00/497M [00:00<?, ?B/s]

train-00001-of-00013.parquet:   0%|          | 0.00/501M [00:00<?, ?B/s]

train-00002-of-00013.parquet:   0%|          | 0.00/495M [00:00<?, ?B/s]

train-00003-of-00013.parquet:   0%|          | 0.00/517M [00:00<?, ?B/s]

...

train-00009-of-00013.parquet:   0%|          | 0.00/532M [00:00<?, ?B/s]

train-00010-of-00013.parquet:   0%|          | 0.00/527M [00:00<?, ?B/s]

train-00011-of-00013.parquet:   0%|          | 0.00/526M [00:00<?, ?B/s]

train-00012-of-00013.parquet:   0%|          | 0.00/543M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15803 [00:00<?, ? examples/s]

In [42]:
np.array(dataset['train']['image']).shape

(15803, 512, 512, 3)

In [26]:
def downsample(example, resolution):
    image = example['image']
    label = example['label']
    
    # Area method with antialias for power-of-2 downsampling
    image = tf.image.resize(image, [resolution, resolution], 
                           method="area", antialias=True)

    return {'image': image, 'label': label}

for resolution in [64, 128, 256]:
    print(f"Processing resolution {resolution}...")
    
    ds_train_res = ds_train.map(lambda x: downsample(x, resolution))
    tf.data.experimental.save(ds_train_res, f"{data_dir}/afhq{resolution}")
    
    print(f"Saved AFHQ-{resolution}")

print("Done! Load with: tf.data.experimental.load(f'{data_dir}/afhq256')")

Processing resolution 64...
Instructions for updating:
Use `tf.data.Dataset.save(...)` instead.


Instructions for updating:
Use `tf.data.Dataset.save(...)` instead.


Saved AFHQ-64
Processing resolution 128...
Saved AFHQ-256
Done! Load with: tf.data.experimental.load(f'{data_dir}/afhq256')


In [29]:
test_ds = tf.data.experimental.load(f'{data_dir}/afhq128').as_numpy_iterator()